In [ ]:
# Installation des bibliothèques
!pip install kaggle pyspark gradio -q
print("✅ Bibliothèques installées !")


✅ Bibliothèques installées !


In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Drive connecté + Modèle chargé !")

Mounted at /content/drive


✅ Drive connecté + Modèle chargé !


In [ ]:
import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

In [ ]:
!pip install pyspark -q
print("✅ PySpark installé !")


In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

+--------------------+----------------+----------------+
|        chemin_image|   classe_reelle|  classe_predite|
+--------------------+----------------+----------------+
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateD

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': ' Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': ' Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': ' Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': ' Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return " Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button(" Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_11393/1224663334.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da8039720ed514c489.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Drive connecté + Modèle chargé !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ Drive connecté + Modèle chargé !


# Nouvelle section

In [ ]:

import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset
License(s): GNU Lesser General Public License 3.0
✅ Dataset prêt !


In [ ]:
!pip install pyspark -q
print("✅ PySpark installé !")

✅ PySpark installé !


In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

+--------------------+----------------+----------------+
|        chemin_image|   classe_reelle|  classe_predite|
+--------------------+----------------+----------------+
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateD

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': ' Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': ' Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': ' Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': ' Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return " Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button(" Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_19928/1224663334.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a34bd3c43f38c2fb9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Déposer le projet sur GitHub

Pour sauvegarder votre projet sur GitHub, vous devez suivre ces étapes. Assurez-vous d'avoir un compte GitHub et, idéalement, un [Personal Access Token (PAT)](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token) pour l'authentification, car l'authentification par mot de passe n'est plus supportée pour les opérations Git sur GitHub.

### 1. Configurer Git

In [7]:
# Configurez votre nom d'utilisateur et votre email pour Git
!git config --global user.email "celia.mazouzi@se.univ-bejaia.dz "
!git config --global user.name "Your Name"

print("✅ Git configuré !")
# N'oubliez pas de remplacer 'your_email@example.com' et 'Your Name' par vos informations GitHub.

✅ Git configuré !


### 2. Initialiser un dépôt Git local

Nous allons initialiser un dépôt Git dans le répertoire de votre projet (généralement `/content/` dans Colab, ou un sous-dossier si vous y avez organisé votre travail).

In [8]:
# Initialiser le dépôt Git (si ce n'est pas déjà fait)
!git init

print("✅ Dépôt Git initialisé !")

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
✅ Dépôt Git initialisé !


### 3. Ajouter et valider les fichiers

Maintenant, nous allons ajouter tous les fichiers du projet au dépôt Git et les valider. Cela inclut le notebook lui-même, les modèles, les données, etc.

In [9]:
# Ajouter tous les fichiers au dépôt
!git add .

# Valider les fichiers
!git commit -m "Initial commit of the Alzheimer Detection project"

print("✅ Fichiers ajoutés et validés !")

error: open("drive/MyDrive/04-Intelligence artificielle-Recherche heuristique-Chapitre 3.gdoc"): Operation not supported
error: unable to index file 'drive/MyDrive/04-Intelligence artificielle-Recherche heuristique-Chapitre 3.gdoc'
fatal: adding files failed
On branch master

Initial commit

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	.gradio/
	alzheimer_data/
	augmented-alzheimer-mri-dataset.zip
	drive/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)
✅ Fichiers ajoutés et validés !


### 4. Créer un dépôt distant sur GitHub

**Action requise de votre part :**

1.  Allez sur [GitHub](https://github.com/).
2.  Connectez-vous à votre compte.
3.  Cliquez sur le bouton **'+ New'** ou **'Create repository'**.
4.  Donnez un nom à votre dépôt (par exemple, `Projet_Alzheimer`).
5.  **Ne cochez PAS** l'option pour ajouter un fichier `README`, `.gitignore` ou une licence (nous allons pousser nos propres fichiers).
6.  Cliquez sur **'Create repository'**.

Une fois le dépôt créé, GitHub vous montrera des instructions. Vous aurez besoin de l'URL du dépôt (par exemple, `https://github.com/votre_nom_utilisateur/Projet_Alzheimer.git`).

### 5. Lier le dépôt local au dépôt distant et pousser les modifications

Utilisez les commandes suivantes pour lier votre dépôt local au dépôt que vous venez de créer sur GitHub et pousser vos fichiers.

**Important :** Remplacez `[YOUR_GITHUB_USERNAME]` par votre nom d'utilisateur GitHub et `[YOUR_REPOSITORY_NAME]` par le nom de votre dépôt. Si vous utilisez un PAT, l'URL sera de la forme `https://[YOUR_PAT]@github.com/[YOUR_GITHUB_USERNAME]/[YOUR_REPOSITORY_NAME].git`.

In [10]:
# Ajoutez l'URL de votre dépôt GitHub distant
# Utilisez un Personal Access Token (PAT) pour l'authentification
# Créez une variable de secret Colab nommée GITHUB_TOKEN pour stocker votre PAT de manière sécurisée.
# Remplacez [YOUR_GITHUB_USERNAME] et [YOUR_REPOSITORY_NAME] par vos informations.

from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') # Assurez-vous d'avoir configuré ce secret

github_username = "[YOUR_GITHUB_USERNAME]" # <--- REMPLACEZ CECI
github_repo_name = "[YOUR_REPOSITORY_NAME]" # <--- REMPLACEZ CECI

remote_url = f"https://{GITHUB_TOKEN}@github.com/{github_username}/{github_repo_name}.git"

!git remote add origin {remote_url}

# Poussez les modifications vers GitHub
!git push -u origin main

print("✅ Projet poussé sur GitHub !")

SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

Si vous rencontrez des problèmes, assurez-vous que votre PAT a les permissions nécessaires (scope `repo`) et que l'URL de votre dépôt est correcte. Vous devrez peut-être aussi configurer votre branche principale (`main` ou `master`) si ce n'est pas déjà fait.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset
!unzip -q augmented-alzheimer-mri-dataset.zip
print(' Dataset téléchargé')
# Structure du dataset
print('\n Structure :')

# New Section

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': ' Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': ' Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': ' Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': ' Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': ' Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': ' Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': ' Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': ' Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return " Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button(" Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_4503/1224663334.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4bae848cc24a43a1f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/alzheimer_model.h5')
print(" Drive connecté + Modèle chargé !")

In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/alzheimer_model (1).h5')
print(" Drive connecté + Modèle chargé !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ Drive connecté + Modèle chargé !


In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/alzheimer_model (1).h5')
print("Drive connecté + Modèle chargé !")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


✅ Drive connecté + Modèle chargé !


In [ ]:
import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print(" Dataset prêt !")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset
License(s): GNU Lesser General Public License 3.0
✅ Dataset prêt !


In [ ]:
!pip install pyspark -q
print("PySpark installé !")

✅ PySpark installé !


In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print(" Pipeline Spark + CNN terminé !")


+--------------------+----------------+----------------+
|        chemin_image|   classe_reelle|  classe_predite|
+--------------------+----------------+----------------+
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateD

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image


classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': ' Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': 'Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': ' Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': 'Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return "Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button(" Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label="Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label=" Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n*Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_2279/2048545935.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8555e99feb6a1d2431.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset
License(s): GNU Lesser General Public License 3.0
✅ Dataset prêt !


In [ ]:
!pip install pyspark -q
print(" PySpark installé !")

✅ PySpark installé !


In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

+--------------------+----------------+----------------+
|        chemin_image|   classe_reelle|  classe_predite|
+--------------------+----------------+----------------+
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|    MildDemented|    MildDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateDemented|ModerateDemented|
|alzheimer_data/Au...|ModerateD

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': ' Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': ' Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': ' Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': ' Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return " Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", "Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button(" Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label=" Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_5136/2048545935.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://969f61c3be03ee9137.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Drive connecté + Modèle chargé !")


In [ ]:
import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)
# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

In [ ]:


Cellule 3 — Installer PySpark :
!pip install pyspark -q
print("✅ PySpark installé !")

In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': ' Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': ' Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': ' Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': ' Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return " Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f"{classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label=" Image IRM à analyser", height=300)
            btn = gr.Button("Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label=" Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(" Drive connecté !")


In [ ]:
  from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Modèle chargé !")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return "⚠️ Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return (" Cette image n'est pas une IRM cérébrale !", " Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f" {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("#  Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Image IRM à analyser", height=300)
            btn = gr.Button("🔍 Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label=" Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label=" Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label=" Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label=" Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

In [ ]:

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Drive connecté + Modèle chargé !")

In [ ]:

import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

In [ ]:
!pip install pyspark -q
print("✅ PySpark installé !")


In [ ]:

from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

In [ ]:
from google.colab import drive
from tensorflow.keras.models import load_model

drive.mount('/content/drive')
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Drive connecté + Modèle chargé !")

In [ ]:
import os
from google.colab import files

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
files.upload()  # Uploade kaggle.json
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

# Télécharger et dézipper
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset prêt !")

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/Projet_Alzheimer'

if os.path.exists(folder_path):
    print(f"✅ Le dossier '{os.path.basename(folder_path)}' existe bien dans votre Google Drive.")
else:
    print(f"❌ Le dossier '{os.path.basename(folder_path)}' n'a pas été trouvé à l'emplacement '{folder_path}'.")
    print("Veuillez vérifier le chemin ou créer le dossier si nécessaire.")

In [ ]:
!pip install pyspark -q
print("✅ PySpark installé !")

In [ ]:
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import os

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:
        data.append((os.path.join(dossier, img_file), classe))

df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))
df_resultats.show(20)
print("✅ Pipeline Spark + CNN terminé !")

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return "⚠️ Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return ("❌ Cette image n'est pas une IRM cérébrale !", "🚫 Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f"🧠 {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("# 🧠 Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="📁 Image IRM à analyser", height=300)
            btn = gr.Button("🔍 Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label="🧠 Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label="⚠️ Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label="💊 Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="📊 Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n⚠️ *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive connecté !")

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5')
print("✅ Modèle chargé !")

In [ ]:
!pip install pyspark -q
print("✅ PySpark installé !")

In [ ]:
# Pipeline complet : Spark + CNN ensemble
from pyspark.sql import SparkSession
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

spark = SparkSession.builder \
    .appName("Alzheimer_Hadoop_Deployment") \
    .getOrCreate()

# Charger les chemins d'images avec Spark
data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

import os
data = []
for classe in classes:
    dossier = os.path.join(data_dir, classe)
    for img_file in os.listdir(dossier)[:10]:  # 10 images par classe
        data.append((os.path.join(dossier, img_file), classe))

# Créer DataFrame Spark
df = spark.createDataFrame(data, ["chemin_image", "classe_reelle"])

# Fonction de prédiction via Spark
def predire_image(chemin):
    img = Image.open(chemin).resize((128, 128))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    return classes[np.argmax(prediction)]

# Appliquer la prédiction sur tout le DataFrame
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

predire_udf = udf(predire_image, StringType())
df_resultats = df.withColumn("classe_predite", predire_udf(df.chemin_image))

# Afficher les résultats
df_resultats.show(20)
print("✅ Déploiement Hadoop + CNN terminé !")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive connecté !")

In [ ]:
from tensorflow.keras.models import load_model
from google.colab import drive
import os

# Ensure Google Drive is mounted.
# Using force_remount=True can help if the mount state is inconsistent.
drive.mount('/content/drive', force_remount=True)
print("✅ Drive connecté !")

model_path = '/content/drive/MyDrive/Projet_Alzheimer/alzheimer_model.h5'

if os.path.exists(model_path):
    model = load_model(model_path)
    print("✅ Modèle chargé !")
else:
    print(f"❌ Erreur : Le fichier du modèle n'a pas été trouvé à l'emplacement : {model_path}")
    print("Veuillez vérifier que le fichier 'alzheimer_model.h5' se trouve bien dans le dossier 'Projet_Alzheimer' de votre Google Drive.")

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return "⚠️ Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return ("❌ Cette image n'est pas une IRM cérébrale !", "🚫 Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f"🧠 {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("# 🧠 Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="📁 Image IRM à analyser", height=300)
            btn = gr.Button("🔍 Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label="🧠 Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label="⚠️ Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label="💊 Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="📊 Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n⚠️ *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive connecté !")

ValueError: mount failed

In [ ]:
import os
from google.colab import files

# Uploader kaggle.json
print("📁 Uploade ton fichier kaggle.json...")
files.upload()

# Configurer Kaggle
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/content/kaggle.json")
os.chmod("/content/kaggle.json", 600)
print("✅ Kaggle configuré !")


In [ ]:
from tensorflow.keras.models import load_model
model = load_model('/content/alzheimer_model.h5')
print("✅ Modèle chargé !")

In [ ]:
# Télécharger le dataset
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset -q
print("✅ Dataset téléchargé !")

# Dézipper
import zipfile
with zipfile.ZipFile("augmented-alzheimer-mri-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("alzheimer_data")
print("✅ Dataset dézippé !")

# Vérifier la structure
import os
for root, dirs, files in os.walk("alzheimer_data"):
    level = root.replace("alzheimer_data", "").count(os.sep)
    if level < 2:
        print(f"{'  ' * level}📁 {os.path.basename(root)}/")


In [11]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image

classes = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
explications = {
    'NonDemented': {'risque': '🟢 Risque FAIBLE', 'explication': 'Aucun signe de démence détecté.', 'conseil': '✅ Continuez les bilans médicaux réguliers.'},
    'VeryMildDemented': {'risque': '🟡 Risque LÉGER', 'explication': 'Signes très légers de démence détectés.', 'conseil': '⚠️ Consultez un médecin pour un suivi préventif.'},
    'MildDemented': {'risque': '🟠 Risque MODÉRÉ', 'explication': 'Signes modérés de démence détectés.', 'conseil': '🔶 Consultation médicale recommandée rapidement.'},
    'ModerateDemented': {'risque': '🔴 Risque ÉLEVÉ', 'explication': 'Signes avancés de démence détectés.', 'conseil': '🚨 Consultation neurologique urgente recommandée.'}
}

def est_irm(img):
    img_array = np.array(img.convert('L'))
    pixels_sombres = np.sum(img_array < 30) / img_array.size
    centre = img_array[img_array.shape[0]//4:3*img_array.shape[0]//4, img_array.shape[1]//4:3*img_array.shape[1]//4]
    bords = np.mean(img_array)
    centre_moyen = np.mean(centre)
    std = np.std(img_array)
    if pixels_sombres > 0.15 and centre_moyen > bords * 0.8 and std > 20:
        return True
    return False

def predire(img):
    if img is None:
        return "⚠️ Veuillez uploader une image !", "", "", {}
    if not est_irm(img):
        return ("❌ Cette image n'est pas une IRM cérébrale !", "🚫 Image invalide", "Veuillez uploader une vraie image IRM cérébrale.", {})
    img_resized = img.resize((128, 128))
    img_array = keras_image.img_to_array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    classe_predite = classes[np.argmax(prediction)]
    confiance = np.max(prediction) * 100
    info = explications[classe_predite]
    resultats = {c: float(prediction[0][i]) for i, c in enumerate(classes)}
    return f"🧠 {classe_predite} — Confiance : {confiance:.2f}%", info['risque'], f"{info['explication']}\n\n{info['conseil']}", resultats

with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:
    gr.Markdown("# 🧠 Système de Détection Précoce de la Maladie d'Alzheimer\n### Uploadez une image IRM cérébrale pour obtenir un diagnostic automatique basé sur l'IA\n---")
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="📁 Image IRM à analyser", height=300)
            btn = gr.Button("🔍 Analyser l'image", variant="primary", size="lg")
            gr.Markdown("*Formats acceptés : JPG, PNG, WEBP*")
        with gr.Column(scale=1):
            diagnostic_output = gr.Textbox(label="🧠 Diagnostic", lines=2, interactive=False)
            risque_output = gr.Textbox(label="⚠️ Niveau de Risque", lines=1, interactive=False)
            explication_output = gr.Textbox(label="💊 Explication & Conseil médical", lines=4, interactive=False)
            proba_output = gr.Label(label="📊 Probabilités par classe", num_top_classes=4)
    gr.Markdown("---\n⚠️ *Ce système est un outil d'aide au diagnostic. Il ne remplace pas l'avis d'un médecin spécialiste.*")
    btn.click(fn=predire, inputs=image_input, outputs=[diagnostic_output, risque_output, explication_output, proba_output])

interface.launch(share=True)

/tmp/ipykernel_19928/2048545935.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Détection Alzheimer") as interface:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6acf8acc3e091fe5b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import numpy as np
from PIL import Image

# Créer le dossier NonIRM
non_irm_dir = "alzheimer_data/AugmentedAlzheimerDataset/NonIRM"
os.makedirs(non_irm_dir, exist_ok=True)

# Type 1 — Images de bruit aléatoire (500 images)
for i in range(500):
    random_img = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
    Image.fromarray(random_img).save(f"{non_irm_dir}/noise_{i:04d}.jpg")

# Type 2 — Images de couleurs solides (200 images)
for i in range(200):
    color = tuple(np.random.randint(0, 255, 3).tolist())
    img = Image.new('RGB', (128, 128), color=color)
    img.save(f"{non_irm_dir}/color_{i:04d}.jpg")

# Type 3 — Images avec formes géométriques (300 images)
for i in range(300):
    img_array = np.zeros((128, 128, 3), dtype=np.uint8)
    # Ajouter des rectangles aléatoires
    for _ in range(5):
        x1, y1 = np.random.randint(0, 100, 2)
        x2, y2 = x1 + np.random.randint(10, 50), y1 + np.random.randint(10, 50)
        color = np.random.randint(0, 255, 3)
        img_array[y1:y2, x1:x2] = color
    Image.fromarray(img_array).save(f"{non_irm_dir}/shapes_{i:04d}.jpg")

total = len(os.listdir(non_irm_dir))
print(f"✅ {total} images Non-IRM créées !")
print("📊 Distribution finale du dataset :")

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
for classe in os.listdir(data_dir):
    nb = len(os.listdir(f"{data_dir}/{classe}"))
    print(f"   {classe}: {nb} images")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = "alzheimer_data/AugmentedAlzheimerDataset"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Entraînement avec augmentation de couleurs
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    channel_shift_range=50.0,    # Changement de couleur ✅
    brightness_range=[0.5, 1.5], # Luminosité variable ✅
    horizontal_flip=True,
    zoom_range=0.2,
    rotation_range=10
)

# Validation sans augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Données d'entraînement
train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Données de validation
val_data = val_datagen.flow_from_directory(
    data_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("✅ Données prêtes !")
print(f"Classes : {train_data.class_indices}")


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

model = Sequential([
    # Bloc 1
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Bloc 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Bloc 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Bloc 4
    Conv2D(256, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Classificateur
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.3),

    # 5 classes
    Dense(5, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("✅ Modèle CNN amélioré créé !")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     4,719,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,242,053 (20.00 MB)

 Trainable params: 5,241,093 (19.99 MB)

 Non-trainable params: 960 (3.75 KB)

✅ Modèle CNN amélioré créé !


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks pour optimiser l'entraînement
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

# Entraînement
history = model.fit(
    train_data,
    epochs=15,
    validation_data=val_data,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("✅ Entraînement terminé !")
